In [1]:
from google.colab import files
uploaded = files.upload()

Saving admissions.csv to admissions.csv
Saving diagnoses_icd.csv to diagnoses_icd.csv
Saving patients.csv to patients.csv
Saving procedures_icd.csv to procedures_icd.csv


In [2]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)
os.makedirs("pipeline", exist_ok=True)
os.makedirs("sql", exist_ok=True)
os.makedirs("docs", exist_ok=True)


In [5]:
import shutil

shutil.move("admissions.csv", "data/raw/admissions.csv")
shutil.move("patients.csv", "data/raw/patients.csv")
shutil.move("diagnoses_icd.csv", "data/raw/diagnoses_icd.csv")
shutil.move("procedures_icd.csv", "data/raw/procedures_icd.csv")


'data/raw/procedures_icd.csv'

In [8]:

# ---------------------------------------------------------
# STEP 1 — IMPORT LIBRARIES
# ---------------------------------------------------------
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# STEP 2 — LOAD EXCEL FILES
# ---------------------------------------------------------
admissions = pd.read_csv("data/raw/admissions.csv")
patients = pd.read_csv("data/raw/patients.csv")
diagnoses = pd.read_csv("data/raw/diagnoses_icd.csv")
procedures = pd.read_csv("data/raw/procedures_icd.csv")

print("Files loaded successfully")

# ---------------------------------------------------------
# STEP 3 — CLEAN BASIC FIELDS
# ---------------------------------------------------------
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])

# Compute LOS if missing
if 'los' not in admissions.columns:
    admissions['los'] = (admissions['dischtime'] - admissions['admittime']).dt.total_seconds() / (24*3600)

admissions = admissions[admissions['los'] > 0]

# ---------------------------------------------------------
# STEP 4 — MERGE ADMISSIONS + PATIENTS
# ---------------------------------------------------------
df = admissions.merge(
    patients[['subject_id', 'gender', 'anchor_age']],
    on='subject_id',
    how='left'
)

df.rename(columns={'anchor_age': 'age'}, inplace=True)
df['age'] = df['age'].clip(0, 100)

# ---------------------------------------------------------
# STEP 5 — ADD FIRST DIAGNOSIS PER ADMISSION
# ---------------------------------------------------------
diag_first = (
    diagnoses
    .sort_values(['hadm_id'])
    .groupby('hadm_id')
    .first()
    .reset_index()
    [['hadm_id', 'icd_code']]
    .rename(columns={'icd_code': 'diagnosis_code'})
)

df = df.merge(diag_first, on='hadm_id', how='left')

# ---------------------------------------------------------
# STEP 6 — ADD FIRST PROCEDURE PER ADMISSION
# ---------------------------------------------------------
proc_first = (
    procedures
    .sort_values(['hadm_id'])
    .groupby('hadm_id')
    .first()
    .reset_index()
    [['hadm_id', 'icd_code']]
    .rename(columns={'icd_code': 'procedure_code'})
)

df = df.merge(proc_first, on='hadm_id', how='left')

# ---------------------------------------------------------
# STEP 7 — FINAL CLEANING
# ---------------------------------------------------------
df['diagnosis_code'] = df['diagnosis_code'].fillna("UNKNOWN")
df['procedure_code'] = df['procedure_code'].fillna("NONE")

df_out = df[[
    'subject_id',
    'hadm_id',
    'gender',
    'age',
    'admission_type',
    'admittime',
    'dischtime',
    'los',
    'diagnosis_code',
    'procedure_code'
]]

# ---------------------------------------------------------
# STEP 8 — SAVE OUTPUT
# ---------------------------------------------------------
df_out.to_csv("data/raw/mimic_los_extract.csv", index=False)

print("Dataset created successfully → data/raw/mimic_los_extract.csv")


Files loaded successfully
Dataset created successfully → data/raw/mimic_los_extract.csv


In [9]:
from google.colab import files
files.download("data/raw/mimic_los_extract.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>